<a href="https://colab.research.google.com/github/Nathanamarante/ResistoMap/blob/main/ResistoMap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResistoMap

---

Objetivo: Mapeamento de genes de resistência a antibióticos presentes na Salmonella enterica

Etapas para elaboração:

3. Selecionar e obter os dados

CARD (card.mcmaster.ca): banco de referência com sequências e ontologia dos genes de resistência.
NCBI Pathogen Detection / NCBI Datasets: genomas e metadados (espécie, país, ano, fonte do isolado), com genes AMR já anotados em muitos casos.

ResFinder / AMRFinderPlus: bases e ferramentas alternativas para comparação.

Documentem os critérios de seleção (espécie, qualidade do assembly, período).

### *CHECKLIST: X (Baixado CARD (Referência) e NCBI)

4. Preparar o ambiente
Python com Biopython, pandas, numpy, matplotlib/seaborn, scipy e, opcionalmente, networkx e scikit-learn. Usem Jupyter ou Colab e um repositório no GitHub.

### *CHECKLIST: X

5. Pré-processamento e controle de qualidade

Filtrar genomas por completude, número de contigs e metadados faltantes.
Padronizar nomes de espécies, países e datas.
Remover duplicatas.

### *CHECKLIST: A FAZER

6. Identificar os genes de resistência
Há duas abordagens, e pode-se combinar ambas:

Usar ferramentas prontas (RGI do CARD ou AMRFinderPlus) e processar a saída em Python.
Implementar uma busca própria: alinhar ou procurar as sequências do CARD nos genomas (por exemplo, com BLAST local e filtros de identidade e cobertura). Isso reforça o lado algorítmico da disciplina.

### *CHECKLIST: A FAZER

7. Construir a matriz presença/ausência
Montem uma tabela genomas × genes (0/1), a base de todas as análises seguintes.

### *CHECKLIST: A FAZER

8. Análises exploratórias e estatísticas

Prevalência de cada gene e de cada classe de antibiótico.
Número médio de genes de resistência por genoma.
Comparação entre grupos (países, anos, fontes: hospitalar, ambiental, animal), com testes estatísticos como qui-quadrado.
Coocorrência de genes (correlação ou índice de Jaccard) e rede de coocorrência.
Agrupamento hierárquico de genomas por perfil de resistência.

### *CHECKLIST: A FAZER

9. Análise avançada (opcional)

PCA ou UMAP sobre a matriz de perfis.
Modelo de machine learning para prever o fenótipo (resistente/sensível) a partir dos genes, com validação cruzada.
Identificação de genomas multirresistentes (MDR).

### *CHECKLIST: A FAZER

10. Visualização dos resultados
Barras de prevalência, heatmap de presença/ausência, mapa ou série temporal, rede de coocorrência e dendrograma.

### *CHECKLIST: A FAZER

11. Interpretação biológica e limitações
Relacionem os achados com a literatura e com a saúde pública. Discutam vieses: genomas de hospitais estão super-representados, e a presença do gene não garante expressão ou resistência fenotípica.

### *CHECKLIST: A FAZER

12. Documentação e reprodutibilidade
README, notebook organizado, lista de dependências e instruções para reproduzir.

### *CHECKLIST: A FAZER

13. Preparar o seminário
Roteiro sugerido: problema → dados → método → resultados → conclusões e limitações. Reservem tempo para ensaiar e dividam as falas entre os integrantes.

### *CHECKLIST: A FAZER

Critérios de seleção dos dados:
Salmonella enterica

Localização: Brazil

Qualidade do Assembly: Não filtrado (Revisar durante o desenvolvimento)

Período: sem restrição de data

_______________________________________________________________________________
Arquivos do CARD Dataset (Referência) são genomas (FASTA). As variantes no NCBI estão em FNA)

_______________________________________________________________________________

Decisão sobre cada arquivo utilizado:
Espécie diferente de Salmonella enterica =	Excluir

Arquivo vazio ou corrompido =	Excluir

Tamanho muito abaixo de 3,5 Mb =	Excluir ou revisar

País faltando, mas análise será por país =	Excluir ou colocar em grupo “desconhecido”

Ano faltando, mas análise será temporal =	Excluir ou colocar em grupo “desconhecido”

Duplicata exata =	Manter apenas uma

Duplicata quase idêntica =	Escolher a de melhor qualidade

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap


/content/drive/MyDrive/Colab Notebooks/ResistoMap


In [13]:
!pip install biopython pandas numpy

In [14]:
# Biblioteca para bioinformática (manipulação de arquivos FASTA, GenBank, etc.)
import Bio
from Bio import SeqIO, Align

# Bibliotecas para manipulação e análise de dados estruturados
import pandas as pd
import numpy as np

# Bibliotecas para análise estatística e matemática
import scipy
from scipy import stats

# Bibliotecas para visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Bibliotecas opcionais para grafos/redes e Machine Learning
import networkx as nx
import sklearn

# Confirmando as versões instaladas
print("Biopython versão:", Bio.__version__)
print("Pandas versão:", pd.__version__)

Biopython versão: 1.88
Pandas versão: 2.2.3


In [15]:
%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap/NCBI Salmonella Enterica

import pandas as pd #Coloquei novamente mas acredito não precisar chamar Pandas de novo, me corrijam se estiver errado

df = pd.read_csv("NCBI dataset isolates.csv") #Arquivos do CARD (Referência) só estão presentes em FASTA, se necessário converter para CSV

print(df.shape)
print(df.columns)
print(df.head())
print(df.dtypes)
print(df.isna().sum())


/content/drive/MyDrive/Colab Notebooks/ResistoMap/NCBI Salmonella Enterica
(4783, 17)
Index(['#Organism group', 'Strain', 'Isolate identifiers', 'Serovar',
       'Isolate', 'Create date', 'Location', 'Isolation source',
       'Isolation type', 'Food origin', 'SNP cluster', 'Min-same', 'Min-diff',
       'BioSample', 'Assembly', 'AMR genotypes', 'Computed types'],
      dtype='object')
       #Organism group       Strain  \
0  Salmonella enterica   IVB 126/82   
1  Salmonella enterica   IVB 176/82   
2  Salmonella enterica       287/91   
3  Salmonella enterica  FDAARGOS_54   
4  Salmonella enterica      BR_RS12   

                                 Isolate identifiers      Serovar  \
0  "BCW_1789","CFSAN000575","IVB 126/82","SRS535828"   Heidelberg   
1  "BCW_1843","CFSAN000164","IVB 176/82","SRS535853"  Enteritidis   
2        "001455_GCAT-20090507","287/91","ERS000073"   Gallinarum   
3        "FDA:FDAARGOS_54","FDAARGOS_54","SRS891668"   Heidelberg   
4                             

In [16]:
print(df["species"].value_counts(dropna=False))
print(df["country"].value_counts(dropna=False))
print(df["year"].value_counts(dropna=False))
print(df["source"].value_counts(dropna=False))


KeyError: 'species'

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/ResistoMap

!head -n 20 GCA_001188125.2.FASTA

In [ ]:
fasta_path = "data/genomes/SALM_001.fasta"

In [ ]:
records = list(SeqIO.parse(fasta_path, "fasta"))